# B4c — CT-DSEM Grade 12 (Continuous-Time Dynamic SEM)

## Run modes available

| Mode | Method | Time on Colab | Inference quality |
|------|--------|---------------|-------------------|
| `optim` | ML optimization | ~3 min | Wald CI (MLE) |
| `map_laplace` | MAP + Hessian Laplace | ~5 min | Laplace approx. posterior |
| **`mcmc`** | **True NUTS HMC, 4 chains × 2000 iter** | **~6-10 hours** | **Full Bayesian (recommended)** |

**Setup checklist (chạy 1 lần đầu):**
1. **Runtime → Change runtime type**: keep Python kernel (notebook uses subprocess)
2. Mount Google Drive (cell tiếp theo)
3. Đảm bảo files đã upload lên Drive:
   - `/content/drive/MyDrive/irt_lsem/src/r/b4c_ctdsem.R`
   - `/content/drive/MyDrive/irt_lsem/outputs/b3_theta/theta_trajectory_1pl_grade_12.csv`

## Recommendations

- **First time**: chạy `optim` để verify pipeline (~3 min)
- **Sensitivity check**: chạy `map_laplace` (~5 min)
- **Publication run**: chạy `mcmc` overnight (Colab Pro recommended for stability over long runs)

## 1. Mount Drive & verify files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/irt_lsem'
REQUIRED = [
    f'{BASE}/src/r/b4c_ctdsem.R',
    f'{BASE}/outputs/b3_theta/theta_trajectory_1pl_grade_12.csv',
]
for p in REQUIRED:
    print(f"{'✓' if os.path.exists(p) else '✗'} {p}")

os.chdir(BASE)
print(f"\nWorking dir: {os.getcwd()}")

# Check Colab CPU + RAM
!cat /proc/cpuinfo | grep 'model name' | head -1
!nproc
!free -h | head -2

## 2. Install R packages (chạy 1 lần)

rstan + ctsem cần compilation. Mất ~10-20 phút lần đầu, các lần sau cache lại.

In [ ]:
import subprocess

INSTALL_R = '''
options(repos = "https://cloud.r-project.org")
pkgs <- c("data.table", "rstan", "ctsem")
missing <- pkgs[!sapply(pkgs, requireNamespace, quietly=TRUE)]
if (length(missing) > 0) {
  cat("Installing:", paste(missing, collapse=", "), "\\n")
  install.packages(missing, Ncpus=4)
} else {
  cat("All packages already installed.\\n")
}
for (p in pkgs) cat(p, ":", as.character(packageVersion(p)), "\\n")
'''

proc = subprocess.Popen(
    ['Rscript', '-e', INSTALL_R],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\nExit: {proc.returncode}")

## 3. Quick test — `optim` mode (~3 min)

Run this first to verify everything works.

In [ ]:
import subprocess, os
BASE = '/content/drive/MyDrive/irt_lsem'
os.chdir(BASE)

env = {
    **os.environ,
    'B3_DIR':         f'{BASE}/outputs/b3_theta',
    'B4_DIR':         f'{BASE}/outputs/b4_lsem',
    'CTDSEM_MODE':    'optim',
    'MIN_T':          '15',
    'N_CORES':        '4',
    'R_PROFILE_USER': '/dev/null',
}

print(f"=== Quick test (optim mode) ===\n")
proc = subprocess.Popen(
    ['Rscript', 'src/r/b4c_ctdsem.R'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\n=== Exit: {proc.returncode} ===")

## 4. PUBLICATION RUN — full MCMC (true NUTS HMC)

**⚠️ Warning**: 6-10 giờ trên Colab. Khuyến nghị Colab Pro để tránh disconnect.

Tips:
- **Free Colab**: idle timeout 90 min, max session 12h. Cần keep tab active hoặc Pro.
- **Save fit thường xuyên**: script đã save vào Drive, nếu disconnect thì pickup được.
- **Trickle progress**: Stan in progress every ~10% per chain.

**Trick để keep Colab session alive**: paste vào Console (F12):
```javascript
function ConnectButton(){
    console.log('Connect pushed');
    document.querySelector('#top-toolbar > colab-connect-button').shadowRoot.querySelector('#connect').click()
}
setInterval(ConnectButton, 60000);
```

In [ ]:
import subprocess, os, time
BASE = '/content/drive/MyDrive/irt_lsem'
os.chdir(BASE)

env = {
    **os.environ,
    'B3_DIR':         f'{BASE}/outputs/b3_theta',
    'B4_DIR':         f'{BASE}/outputs/b4_lsem',
    'CTDSEM_MODE':    'mcmc',          # ← TRUE HMC
    'MIN_T':          '15',
    'N_CORES':        '4',
    'R_PROFILE_USER': '/dev/null',
}

print(f"=== Full MCMC run (NUTS, 4 chains × 2000 iter) ===\n")
print(f"Started at: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("Expected: 6-10 hours. Save fit will be at outputs/b4_lsem/ctdsem_grade_12.rds\n")

t0 = time.time()
proc = subprocess.Popen(
    ['Rscript', 'src/r/b4c_ctdsem.R'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True, env=env
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = (time.time() - t0) / 60
print(f"\n=== Exit: {proc.returncode} | Total: {elapsed:.1f} min ===")

## 5. Verify outputs & check MCMC diagnostics

In [ ]:
# List outputs
import os
OUT = '/content/drive/MyDrive/irt_lsem/outputs/b4_lsem'
for f in sorted(os.listdir(OUT)):
    if 'ctdsem' in f.lower():
        size = os.path.getsize(f'{OUT}/{f}') / 1024
        print(f'  {f:50s} {size:>8.1f} KB')

print('\n=== ctdsem_summary.txt ===')
with open(f'{OUT}/ctdsem_summary.txt') as fh:
    print(fh.read())

In [ ]:
# Run MCMC diagnostics: R-hat, ESS, divergences
import subprocess

DIAG_R = '''
suppressPackageStartupMessages({library(ctsem); library(rstan)})
fit <- readRDS("outputs/b4_lsem/ctdsem_grade_12.rds")

if ("stanfit" %in% names(fit)) {
  sf <- fit$stanfit
  cat("=== MCMC DIAGNOSTICS ===\\n")
  cat("Chains:", sf@sim$chains, "| Iter per chain:", sf@sim$iter, "\\n")
  cat("Warmup:", sf@sim$warmup, "\\n\\n")
  
  smry <- rstan::summary(sf)$summary
  cat("R-hat range:    [", round(min(smry[,"Rhat"], na.rm=TRUE), 3), ", ",
      round(max(smry[,"Rhat"], na.rm=TRUE), 3), "]\\n", sep="")
  cat("R-hat > 1.05:   ", sum(smry[,"Rhat"] > 1.05, na.rm=TRUE), " / ", nrow(smry), "\\n", sep="")
  cat("ESS bulk min:   ", round(min(smry[,"n_eff"], na.rm=TRUE)), "\\n", sep="")
  cat("ESS bulk < 400: ", sum(smry[,"n_eff"] < 400, na.rm=TRUE), " / ", nrow(smry), "\\n", sep="")
  
  divs <- get_num_divergent(sf)
  treedepth <- get_num_max_treedepth(sf)
  cat("Divergent transitions:", divs, "\\n")
  cat("Max treedepth hits:   ", treedepth, "\\n\\n")
  
  cat("=== POPULATION POSTERIOR ===\\n")
  pops <- c("pop_phi", "pop_cint", "pop_sigma", "pop_manifest_var",
            "pop_var_T0", "pop_mean_T0")
  pop_rows <- rownames(smry)[grepl(paste(pops, collapse="|"), rownames(smry))]
  if (length(pop_rows) > 0) print(round(smry[pop_rows, c("mean","sd","2.5%","50%","97.5%","Rhat","n_eff")], 4))
} else {
  cat("This fit was not produced by MCMC mode.\\n")
}
'''

proc = subprocess.Popen(
    ['Rscript', '-e', DIAG_R],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True
)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

## 6. Sync results back to local

After MCMC completes, the results are saved to Drive. Sync local repo:

```bash
# On local machine, pull from Drive sync folder:
cp -r ~/Drive/MyDrive/irt_lsem/outputs/b4_lsem/ctdsem_* \\
     /home/dtnntd/olm_irt/research/irt_lsem/outputs/b4_lsem/

# Re-run B5 consolidate + verify
cd /home/dtnntd/olm_irt/research/irt_lsem
R_PROFILE_USER=/dev/null Rscript src/r/b5_consolidate.R
R_PROFILE_USER=/dev/null Rscript src/r/b5_verify_final.R
```